# Requests
Requirements:
- Use preferrably Python 3.9.6 venvs.
- Add a `.env` file in the same folder as this notebook with an `ANTHROPIC_API_KEY="<api_key_here>"` field.

## 1 Making a request
- API
- Auth

In [1]:
# %pip install anthropic python-dotenv

In [2]:
# Load api key from .env file

from dotenv import load_dotenv

load_dotenv()

from dotenv import dotenv_values

config = dotenv_values(".env")
auth_token = config.get("ANTHROPIC_API_KEY")

In [3]:
# Auth and model

from anthropic import Anthropic

client = Anthropic(
    auth_token=auth_token,
    base_url="https://flow.ciandt.com/flow-llm-proxy"
)

model = "bedrock/anthropic.claude-4-6-sonnet"
# model = "claude-sonnet-4-0"

## 2 Multi-Turn conversations
- Helpers

In [4]:
# Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

## 3 System prompt - Math tutor
- Chat func with optional system prompt

In [5]:
# New chat function

def chat(
    messages,
    system=None,
):
    # Required
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    # Optional
    if system:
        params["system"] = system
    
    # KW-args
    message = client.messages.create(**params)
    return message.content[0].text

In [6]:
messages = [] 

user_prompt = "How do I solve 5x + 2 = 3 for x?"
add_user_message(messages, user_prompt)

In [7]:
# Without system prompt

answer = chat(messages)
print(answer)

## Solving for x

**Starting equation:**
5x + 2 = 3

**Step 1: Subtract 2 from both sides**
$$5x + 2 - 2 = 3 - 2$$
$$5x = 1$$

**Step 2: Divide both sides by 5**
$$\frac{5x}{5} = \frac{1}{5}$$
$$x = \frac{1}{5}$$

---

**Answer: x = 1/5 (or 0.2)**

You can verify this by plugging it back into the original equation:
5(1/5) + 2 = 1 + 2 = 3 ✓


In [8]:
# With system prompt

system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

answer = chat(messages, system=system)
print(answer)

Let's work through this step by step!

The goal is to **get x by itself** on one side of the equation.

**First, let's think about what's "attached" to x:**
- There's a `+2` on the left side
- There's a `5` being multiplied by x

**Step 1: Get rid of the +2 first.**

What could you do to *both sides* of the equation to remove the +2? What's the opposite of adding 2?

Give that a try and tell me what you get! 😊


## 4 System Prompt Exercise

In [8]:
user_prompt = "write a Python function that checks a string for duplicate characters."

In [9]:
# No system prompt 

messages = []

add_user_message(
    messages,
    user_prompt,
)

answer = chat(messages)

print(answer)

## Checking for Duplicate Characters in a String

Here's a Python function that checks a string for duplicate characters:

```python
def check_duplicates(string: str) -> dict:
    """
    Checks a string for duplicate characters.

    Args:
        string: The input string to check.

    Returns:
        A dictionary containing:
        - 'has_duplicates': Boolean indicating if duplicates exist
        - 'duplicates': Dictionary of duplicate characters and their counts
    """
    char_count = {}
    
    for char in string:
        char_count[char] = char_count.get(char, 0) + 1
    
    duplicates = {char: count for char, count in char_count.items() if count > 1}
    
    return {
        "has_duplicates": bool(duplicates),
        "duplicates": duplicates
    }


# Example usage
if __name__ == "__main__":
    test_strings = [
        "hello",
        "world",
        "python",
        "abcdef",
        "mississippi"
    ]

    for s in test_strings:
        result = check_duplicates(

In [11]:
# Concise system prompt

system_prompt = "You should write concise code with no explanations."

messages = []

add_user_message(
    messages,
    user_prompt,
)

answer = chat(
    messages,
    system=system_prompt,
)

print(answer)

```python
def has_duplicate_characters(s: str) -> bool:
    return len(s) != len(set(s))
```

**Example usage:**

```python
print(has_duplicate_characters("hello"))   # True  ('l' appears twice)
print(has_duplicate_characters("world"))   # False
print(has_duplicate_characters("python"))  # False
print(has_duplicate_characters("abcabc")) # True
```

If you also want to **identify which characters are duplicates**, use this version:

```python
def find_duplicate_characters(s: str) -> list:
    seen = set()
    duplicates = set()
    for char in s:
        if char in seen:
            duplicates.add(char)
        seen.add(char)
    return list(duplicates)
```

**Example usage:**

```python
print(find_duplicate_characters("hello"))   # ['l']
print(find_duplicate_characters("abcabc"))  # ['a', 'b', 'c']
print(find_duplicate_characters("world"))   # []
```
